### Loading and Preprocessing Amazon Reviews Dataset

This notebook loads the Amazon Product Reviews Dataset from Google Cloud Storage (GCS), processes the data using PySpark, and saves the cleaned dataset for further sentiment analysis. It includes schema definition, validation, and handling of category-specific files before merging them into a single dataset.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, isnull, mean, stddev, min, max, length, split, element_at, when, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
import pandas as pd

In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Amazon Reviews Data") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [3]:
# Path to data
data_path = "gs://final-project-bucket-amazon/amazon-reviews/*.tsv"

# Define the schema explicitly based on your column list
schema = StructType([
    StructField("marketplace", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("review_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_parent", StringType(), True),
    StructField("product_title", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("star_rating", IntegerType(), True),
    StructField("helpful_votes", IntegerType(), True),
    StructField("total_votes", IntegerType(), True),
    StructField("vine", StringType(), True),
    StructField("verified_purchase", StringType(), True),
    StructField("review_headline", StringType(), True),
    StructField("review_body", StringType(), True),
    StructField("review_date", DateType(), True)
])

# Load the TSV files
reviews_df = spark.read.csv(
    "gs://final-project-bucket-amazon/amazon-reviews/*.tsv", 
    sep="\t", 
    header=True,
    schema=schema,
    quote='"',
    escape='"'
)

In [4]:
# View first few rows of the dataset
print("Sample data from reviews dataset:")
reviews_df.show(5, truncate=False)

Sample data from reviews dataset:
+-----------+-----------+--------------+----------+--------------+------------------------------------------------------------------------------------+----------------+-----------+-------------+-----------+----+-----------------+-------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Count rows in combined DataFrame
total_rows = reviews_df.count()
print(f"Total rows in combined DataFrame: {total_rows}")

# Check how many rows came from each file
file_counts = reviews_df.withColumn("category", col("product_category")).groupBy("category").count()
file_counts.show()

Total rows in combined DataFrame: 18118476
+--------------------+-------+
|            category|  count|
+--------------------+-------+
|                null|    246|
|  Mobile_Electronics| 104963|
|             Apparel|5906322|
|          2003-02-05|      1|
|I don't get this ...|      1|
|         Electronics|3093861|
|               Books|3105515|
|          2002-08-07|      1|
|           Furniture| 792113|
|          2005-03-11|      1|
|              Beauty|5115452|
+--------------------+-------+



From the above output, we noticed that the TSV files weren't being correctly parsed due to formatting inconsistencies across the dataset. The tab delimiters within quoted fields and embedded line breaks were causing columns to misalign. To address this issue, we implemented a custom parsing approach that reads each file as raw text first, then properly splits the columns using delimiters. We processed each category file individually, applied the corrections, and saved them as separate Parquet files. Finally, we concatenated all the cleaned datasets into a single comprehensive dataset containing over 18 million reviews across all product categories. This combined dataset is now saved in the Parquet format, which will significantly improve loading times for our subsequent analysis

In [ ]:
def parse_amazon_reviews_file(file_path, expected_category):
    """
    Parse the Amazon reviews TSV file with custom handling to avoid parsing issues.
    
    Args:
        file_path: Path to the TSV file in GCS
        expected_category: The category this file should contain
        
    Returns:
        A properly parsed DataFrame
    """
    print(f"Processing {file_path}...")
    
    # Load the file as a single column text file
    raw_df = spark.read.text(file_path)
    
    # Get and skip the header row
    header = raw_df.first()[0]
    data_df = raw_df.filter(col("value") != header)
    
    # Split the text by tabs
    split_cols = split(data_df.value, "\t")
    
    # Create a new DataFrame with the correct columns
    parsed_df = data_df.select(
        element_at(split_cols, 1).alias("marketplace"),
        element_at(split_cols, 2).alias("customer_id"),
        element_at(split_cols, 3).alias("review_id"),
        element_at(split_cols, 4).alias("product_id"),
        element_at(split_cols, 5).alias("product_parent"),
        element_at(split_cols, 6).alias("product_title"),
        element_at(split_cols, 7).alias("product_category"),
        element_at(split_cols, 8).cast("int").alias("star_rating"),
        element_at(split_cols, 9).cast("int").alias("helpful_votes"),
        element_at(split_cols, 10).cast("int").alias("total_votes"),
        element_at(split_cols, 11).alias("vine"),
        element_at(split_cols, 12).alias("verified_purchase"),
        element_at(split_cols, 13).alias("review_headline"),
        element_at(split_cols, 14).alias("review_body"),
        element_at(split_cols, 15).cast("date").alias("review_date")
    )
    
    # Fix any category issues by setting expected category
    parsed_df = parsed_df.withColumn(
        "product_category", 
        when(
            (col("product_category").isNull()) | 
            (col("product_category") != expected_category),
            lit(expected_category)
        ).otherwise(col("product_category"))
    )
    
    # Add helpful ratio column
    parsed_df = parsed_df.withColumn(
        "helpful_ratio", 
        when(col("total_votes") > 0, col("helpful_votes") / col("total_votes")).otherwise(None)
    )
    
    # Run some validation checks
    total_rows = parsed_df.count()
    null_categories = parsed_df.filter(col("product_category").isNull()).count()
    wrong_categories = parsed_df.filter(col("product_category") != expected_category).count()
    
    print(f"  Total rows: {total_rows}")
    print(f"  Null categories: {null_categories}")
    print(f"  Wrong categories: {wrong_categories}")
    
    # Return the parsed DataFrame
    return parsed_df

# Define the files and their expected categories
file_categories = [
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Apparel_v1_00.tsv", "Apparel"),
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Beauty_v1_00.tsv", "Beauty"),
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Books_v1_02.tsv", "Books"),
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Electronics_v1_00.tsv", "Electronics"),
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Furniture_v1_00.tsv", "Furniture"),
    ("gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Mobile_Electronics_v1_00.tsv", "Mobile_Electronics")
]

# Process each file individually and store in a dictionary
parsed_dfs = {}

for file_path, category in file_categories:
    category_key = category.lower().replace("_", "")
    df = parse_amazon_reviews_file(file_path, category)
    
    # Save each individual DataFrame to parquet for checkpointing
    output_path = f"gs://final-project-bucket-amazon/processed/{category_key}_reviews.parquet"
    df.write.mode("overwrite").parquet(output_path)
    print(f"  Saved to {output_path}")
    
    # Store reference in dictionary
    parsed_dfs[category] = df

# Combine all DataFrames
print("\nCombining all DataFrames...")
combined_df = None

for category, df in parsed_dfs.items():
    if combined_df is None:
        combined_df = df
    else:
        combined_df = combined_df.union(df)

# Verify the combined dataset
print("Final combined dataset statistics:")
total_rows = combined_df.count()
print(f"Total rows: {total_rows}")

print("Distribution by category:")
combined_df.groupBy("product_category").count().orderBy("count", ascending=False).show()

# Save the final combined dataset
combined_df.write.mode("overwrite").parquet("gs://final-project-bucket-amazon/processed/all_reviews.parquet")
print("Combined dataset saved to gs://final-project-bucket-amazon/processed/all_reviews.parquet")

Processing gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Apparel_v1_00.tsv...
  Total rows: 5906333
  Null categories: 0
  Wrong categories: 0
  Saved to gs://final-project-bucket-amazon/processed/apparel_reviews.parquet
Processing gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Beauty_v1_00.tsv...
  Total rows: 5115666
  Null categories: 0
  Wrong categories: 0
  Saved to gs://final-project-bucket-amazon/processed/beauty_reviews.parquet
Processing gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Books_v1_02.tsv...
  Total rows: 3105520
  Null categories: 0
  Wrong categories: 0
  Saved to gs://final-project-bucket-amazon/processed/books_reviews.parquet
Processing gs://final-project-bucket-amazon/amazon-reviews/amazon_reviews_us_Electronics_v1_00.tsv...
  Total rows: 3093869
  Null categories: 0
  Wrong categories: 0
  Saved to gs://final-project-bucket-amazon/processed/electronics_reviews.parquet
Processing gs://final-project-buck

In [11]:
# Get basic information about the dataset
print("Schema Information:")
reviews_df.printSchema()

Schema Information:
root
 |-- marketplace: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)



In [12]:
spark.stop()